In [1]:
import numpy as np
import pandas as pd
from skforecast.utils import save_forecaster
from skforecast.utils import load_forecaster

In [2]:
import sys
import os
sys.path.append(os.pardir)

In [3]:
# Exogenous features helpers
from features import set_holidays, cal_features, cyclic_features

In [4]:
from datetime import datetime
from datetime import timedelta

In [40]:
from extract import get_weather_client, fetch_weather
from extract import get_load_client, fetch_load, clean_load

In [5]:
forecaster_loaded = load_forecaster('../model/forecaster_001.joblib', verbose=True, suppress_warnings=False)

/Users/maxwellgriffith/miniconda3/envs/loadcast/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ForecasterRecursive 
Estimator: LGBMRegressor 
Lags: [  1   2   3  20  21  22  23  24  25  26  27  28  29 146 167 170] 
Window features: None 
Window size: 170 
Series name: Demand 
Exogenous included: True 
Exogenous names: 
    Temperature, Holiday, week, day_of_week, hour, week_sin, week_cos, hour_sin,
    hour_cos, Temp_3D_Mean, Temp_2D_Max, Temp_2D_Min, Temp_1D_Min 
Categorical features: auto 
Transformer for y: None 
Transformer for exog: None 
Weight function included: False 
Differentiation order: None 
Drop NaN from series: False 
Training range: [Timestamp('2023-01-01 00:00:00'), Timestamp('2024-12-31 23:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <Hour> 
Estimator parameters: 
    {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0,
    'importance_type': 'split', 'learning_rate': 0.16546988609195726,
    'max_depth': 3, 'min_child_samples': 20, 'min_child_weight': 0.001,
    'min_split_gain': 0.0, 'n_estimators': 700, 'n_jobs'

- make regular predictions without retraining model using last_window
- This argument allows providing only the past values needed to create the autoregressive predictors ie lags
- When using the last_window argument, it is crucial to ensure that the length of last_window is sufficient to include the maximum lag (or custom predictor) used by the forecaster. For instance, if the forecaster employs lags 1, 24, and 48, last_window must include the most recent 48 values of the series

In [6]:
forecaster_loaded.last_window_

,Demand
date,
2024-12-24 22:00:00,30801.208
2024-12-24 23:00:00,31410.891
2024-12-25 00:00:00,31305.535
2024-12-25 01:00:00,30949.415
2024-12-25 02:00:00,30649.751
...,...
2024-12-31 19:00:00,32951.913
2024-12-31 20:00:00,32711.893
2024-12-31 21:00:00,32615.968


In [7]:
exo_vars = forecaster_loaded.exog_names_in_
exo_vars

['Temperature',
 'Holiday',
 'week',
 'day_of_week',
 'hour',
 'week_sin',
 'week_cos',
 'hour_sin',
 'hour_cos',
 'Temp_3D_Mean',
 'Temp_2D_Max',
 'Temp_2D_Min',
 'Temp_1D_Min']

In [8]:
#last window is the end of the validation split so we don't need to pull new load from gridstatus to tests the prod for now

In [9]:
df = (pd.read_csv("../data/raw/gsloadtemp_clean.csv")
        .drop_duplicates()
        .pipe(lambda df: df.set_index(pd.to_datetime(df["date"])))
        .drop(columns = ["date"])
     )
#this will throw an error if there are duplicates
df.index = df.index.tz_localize(None)
df.index.freq = 'h'

In [10]:
df.head()

,Demand,Temperature
date,,
2023-01-01 00:00:00,28771.933,6.90
2023-01-01 01:00:00,28488.282,7.70
2023-01-01 02:00:00,28073.965,6.55
2023-01-01 03:00:00,27756.863,5.80
2023-01-01 04:00:00,27271.343,5.65


In [11]:
TEST_START  = "2025-01-01 00:00:00"
TEST_END    = "2025-12-31 23:00:00"

In [12]:
data_test  = df.loc[TEST_START:, :].copy()

In [13]:
print(f"Test dates       : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")

Test dates       : 2025-01-01 00:00:00 --- 2025-12-31 23:00:00  (n=8760)


- years worth of data to test on
- lags 170
- exogenous variables
    - Temperature
    - Holiday
    - week
    - day_of_week
    - hour
    - week_sin
    - week_cos
    - hour_sin
    - hour_cos
    - Temp_3D_Mean
    - Temp_2D_Max
    - Temp_2D_Min
    - Temp_1D_Min 

In [14]:
data_test = (data_test
            .pipe(set_holidays)
            .pipe(cal_features)
            .pipe(cyclic_features)
            .assign(Holiday=lambda d: d["Holiday"].astype(int))
            .assign(Temp_3D_Mean=lambda d: d["Temperature"].rolling("3D", center = False).mean())
            .assign(Temp_2D_Max =lambda d: d["Temperature"].rolling("2D", center = False).max())
            .assign(Temp_2D_Min =lambda d: d["Temperature"].rolling("2D", center = False).min())
            .assign(Temp_1D_Min =lambda d: d["Temperature"].rolling("1D", center = False).min())
           )

In [15]:
data_test.columns

Index(['Demand', 'Temperature', 'Holiday', 'month', 'week', 'day_of_week',
       'hour', 'month_sin', 'month_cos', 'week_sin', 'week_cos', 'day_sin',
       'day_cos', 'hour_sin', 'hour_cos', 'Temp_3D_Mean', 'Temp_2D_Max',
       'Temp_2D_Min', 'Temp_1D_Min'],
      dtype='str')

In [ ]:
data_test[exo_vars]

In [17]:
forecaster_loaded.predict(
    steps = 24,
    exog = data_test[exo_vars]
)

2025-01-01 00:00:00    34266.067496
2025-01-01 01:00:00    34233.980226
2025-01-01 02:00:00    33924.012688
2025-01-01 03:00:00    33377.508707
2025-01-01 04:00:00    32431.177194
2025-01-01 05:00:00    31456.868182
2025-01-01 06:00:00    30908.842679
2025-01-01 07:00:00    30701.304699
2025-01-01 08:00:00    30585.089811
2025-01-01 09:00:00    30794.115410
2025-01-01 10:00:00    31296.782683
2025-01-01 11:00:00    32258.764408
2025-01-01 12:00:00    33559.258594
2025-01-01 13:00:00    34513.908839
2025-01-01 14:00:00    34644.441200
2025-01-01 15:00:00    34367.594721
2025-01-01 16:00:00    34137.029226
2025-01-01 17:00:00    33665.437853
2025-01-01 18:00:00    33200.678524
2025-01-01 19:00:00    32806.524986
2025-01-01 20:00:00    32447.158699
2025-01-01 21:00:00    32213.572052
2025-01-01 22:00:00    32567.169344
2025-01-01 23:00:00    33501.410006
Freq: h, Name: pred, dtype: float64

In [18]:
#now we use "last window"
# need 170 a little over 7 days 
forecaster_loaded.window_size

170

In [19]:
#write function that takes the date you want to start predictions at and gives you the window before it

td170h = pd.Timedelta(170, "hours")
td170h

Timedelta('7 days 02:00:00')

In [20]:
datetime(2025, 2, 1, 0)

datetime.datetime(2025, 2, 1, 0, 0)

In [21]:
datetime(2025, 2, 1, 0) - timedelta(hours=24)

datetime.datetime(2025, 1, 31, 0, 0)

In [22]:
datetime(2025, 2, 1, 0).isoformat(' ')

'2025-02-01 00:00:00'

In [23]:
def get_last_window(window_size_hrs: int, target_date: datetime):
    window_start = target_date - timedelta(hours = window_size_hrs)
    return window_start

In [24]:
get_last_window()

TypeError: get_last_window() missing 2 required positional arguments: 'window_size_hrs' and 'target_date'

- Lets try and predict the first day in march 2024 so "2025-03-01 00" to "2025-03-01 23"

In [ ]:
window_start = datetime(2025, 3, 1, 0)-timedelta(days = 8)
window_start

In [ ]:
last_window_start = window_start.isoformat(' ')
last_window_start

In [ ]:
window_end = datetime(2025, 3, 1, 0)-timedelta(hours = 1)
window_end

In [ ]:
last_window_end = window_end.isoformat(' ')
last_window_end

In [ ]:
lw_data = data_test.loc[last_window_start:last_window_end, ["Demand"]].copy()
lw_data

In [ ]:
exo_predict = data_test.loc["2025-03-01 00:00:00": "2025-03-01 23:00:00", exo_vars].copy()
exo_predict

In [ ]:
forecaster_loaded.predict(
    steps = 24,
    last_window = lw_data,
    exog = exo_predict
)

- Now we need write functions that get all the correct dates for the previous window and the exo variables
- remeber that the most recent load actuals are D-1
- Predicting D+1
    - exogenous variables for D1 and D+1
    - load forecat for D-1

In [ ]:
today_mn = datetime.today().replace(hour = 0, minute = 0, second = 0, microsecond = 0)
today_mn

In [ ]:
today_mn + timedelta(hours = 47)

In [25]:
def get_date_ranges(day_one = None):
    dates = {}
    if day_one is None:
        day_one = datetime.today().replace(hour = 0, minute = 0, second = 0, microsecond = 0)
    else:
        day_one = day_one.replace(hour = 0, minute = 0, second = 0, microsecond = 0)
    
    dates["today"] = day_one
    #D-1 midnight to D-1 11pm
    lw_start = day_one-timedelta(days = 8)
    dates["lw_start"] = lw_start
    lw_end = day_one - timedelta(hours = 1)
    dates["lw_end"] = lw_end
    
    dates["exo_start"] = day_one
    exo_end = day_one + timedelta(hours = 47)
    dates["exo_end"] = exo_end

    return dates

In [ ]:
pred_dates = get_date_ranges()
pred_dates

In [ ]:
{'today': datetime.datetime(2026, 8, 1, 0, 0),
 'lw_start': datetime.datetime(2026, 7, 24, 0, 0),
 'lw_end': datetime.datetime(2026, 7, 31, 23, 0),
 'exo_start': datetime.datetime(2026, 8, 1, 0, 0),
 'exo_end': datetime.datetime(2026, 8, 2, 23, 0)}

In [ ]:
test_date = datetime(2026, 9, 1, 0, 0)
test_date

In [ ]:
test_date_ranges = get_date_ranges(test_date)
test_date_ranges

In [42]:
test_date = datetime(2026, 8, 1, 0,0)
test_date

datetime.datetime(2026, 8, 1, 0, 0)

In [43]:
date_ranges = get_date_ranges(test_date)
date_ranges

{'today': datetime.datetime(2026, 8, 1, 0, 0),
 'lw_start': datetime.datetime(2026, 7, 24, 0, 0),
 'lw_end': datetime.datetime(2026, 7, 31, 23, 0),
 'exo_start': datetime.datetime(2026, 8, 1, 0, 0),
 'exo_end': datetime.datetime(2026, 8, 2, 23, 0)}

- should probably add a check on day_one to make sure it's at midnight

## Assembl the previous window and exogenous variables from the API

In [31]:
#Weather data
weather_client = get_weather_client()

In [ ]:
#Need to re-write to use the forecast URL not the archive URL
weather_data = fetch_weather(
    weather_client, 
    start = date_ranges['exo_start'],
    end = date_ranges['exo_end']
    )

In [36]:
date_ranges["exo_start"]

datetime.datetime(2026, 8, 3, 0, 0)

In [39]:
load_client = get_load_client()

In [44]:
load_data = fetch_load(
    load_client,
    start = date_ranges["lw_start"],
    end = date_ranges["lw_end"],
    write_csv = False)

2026-08-03 14:57:30 - INFO - Fetching Page 1...
2026-08-03 14:57:30 - INFO - GET https://api.gridstatus.io/v1/datasets/spp_load_hourly/query
2026-08-03 14:57:30 - INFO - Params: {'start_time': Timestamp('2026-07-24 00:00:00'), 'end_time': Timestamp('2026-07-31 23:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': None, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': 'control_zone_name', 'filter_value': 'SYSTEM_TOTAL', 'filter_operator': '=', 'columns': 'interval_start_utc,balancing_area_name,control_zone_name,forecast_area_type,load', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-08-03 14:57:31 - INFO - Done in 0.5 seconds. 
2026-08-03 14:57:31 - INFO - Total number of rows: 0


In [56]:
start_date = date_ranges["lw_start"]
end_date = datetime(2026, 7,26,0,0)

In [48]:
load_data = fetch_load(
    load_client,
    start = start_date,
    end = end_date,
    write_csv = False)

2026-08-03 15:02:54 - INFO - Fetching Page 1...
2026-08-03 15:02:54 - INFO - GET https://api.gridstatus.io/v1/datasets/spp_load_hourly/query
2026-08-03 15:02:54 - INFO - Params: {'start_time': Timestamp('2026-07-24 00:00:00'), 'end_time': Timestamp('2026-07-31 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': None, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': 'control_zone_name', 'filter_value': 'SYSTEM_TOTAL', 'filter_operator': '=', 'columns': 'interval_start_utc,balancing_area_name,control_zone_name,forecast_area_type,load', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-08-03 15:02:55 - INFO - Done in 0.49 seconds. 
2026-08-03 15:02:55 - INFO - Total number of rows: 0


In [58]:
def fetch_load_nofilter(client, start: datetime, end: datetime) -> pd.DataFrame:
    df = client.get_dataset(
        "spp_load_hourly",
        start = start.isoformat(),
        end = end.isoformat(),
        columns = ["interval_start_utc", "balancing_area_name", "control_zone_name", "forecast_area_type", "load"],
        limit =24
    )

    return df

In [59]:
load_data = fetch_load_nofilter(
    load_client,
    start = start_date,
    end = end_date,
)

2026-08-03 15:10:54 - INFO - Fetching Page 1...
2026-08-03 15:10:54 - INFO - GET https://api.gridstatus.io/v1/datasets/spp_load_hourly/query
2026-08-03 15:10:54 - INFO - Params: {'start_time': Timestamp('2026-07-24 00:00:00'), 'end_time': Timestamp('2026-07-26 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': 24, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'columns': 'interval_start_utc,balancing_area_name,control_zone_name,forecast_area_type,load', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-08-03 15:10:55 - INFO - Done in 0.53 seconds. 
2026-08-03 15:10:55 - INFO - Total rows: 24/24 (100.0% of limit)
2026-08-03 15:10:55 - INFO - Total number of rows: 24


In [60]:
load_data

,interval_start_utc,interval_end_utc,balancing_area_name,control_zone_name,forecast_area_type,load
0,2026-07-24 00:00:00+00:00,2026-07-24 01:00:00+00:00,SPP,CSWS,CF,8427.483
1,2026-07-24 00:00:00+00:00,2026-07-24 01:00:00+00:00,SPP,EDE,CF,840.198
2,2026-07-24 00:00:00+00:00,2026-07-24 01:00:00+00:00,SPP,GRDA,CF,1409.844
3,2026-07-24 00:00:00+00:00,2026-07-24 01:00:00+00:00,SPP,INDN,CF,141.603
4,2026-07-24 00:00:00+00:00,2026-07-24 01:00:00+00:00,SPP,KACY,CF,289.395
5,2026-07-24 00:00:00+00:00,2026-07-24 01:00:00+00:00,SPP,KCPL,CF,2452.532
6,2026-07-24 00:00:00+00:00,2026-07-24 01:00:00+00:00,SPP,LES,CF,501.188
7,2026-07-24 00:00:00+00:00,2026-07-24 01:00:00+00:00,SPP,LES,NC,99.625
8,2026-07-24 00:00:00+00:00,2026-07-24 01:00:00+00:00,SPP,MPS,CF,1233.610
9,2026-07-24 00:00:00+00:00,2026-07-24 01:00:00+00:00,SPP,NPPD,CF,3281.165


In [62]:
df = load_client.get_dataset(
    "spp_load_hourly",
    start="2026-07-24",
    end="2026-07-25",
)

2026-08-03 16:06:19 - INFO - Fetching Page 1...
2026-08-03 16:06:19 - INFO - GET https://api.gridstatus.io/v1/datasets/spp_load_hourly/query
2026-08-03 16:06:19 - INFO - Params: {'start_time': Timestamp('2026-07-24 00:00:00'), 'end_time': Timestamp('2026-07-25 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': None, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-08-03 16:06:19 - INFO - Done in 0.45 seconds. 
2026-08-03 16:06:19 - INFO - Total number of rows: 624


In [ ]:
#I guess they don't report the "System total" control zone name for the most recent load data?

In [64]:
df_sample = load_client.get_dataset(
    "spp_load_hourly",
    start = datetime(2026, 07, 1).isoformat(),
    end = datetime(2025, 12, 2).isoformat(),
    columns = ["interval_start_utc", "control_zone_name", "forecast_area_type", "load"],
)

2026-08-03 16:26:33 - INFO - Fetching Page 1...
2026-08-03 16:26:33 - INFO - GET https://api.gridstatus.io/v1/datasets/spp_load_hourly/query
2026-08-03 16:26:33 - INFO - Params: {'start_time': Timestamp('2025-12-01 00:00:00'), 'end_time': Timestamp('2025-12-02 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': None, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'columns': 'interval_start_utc,control_zone_name,forecast_area_type,load', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-08-03 16:26:33 - INFO - Done in 0.44 seconds. 
2026-08-03 16:26:33 - INFO - Total number of rows: 432


In [67]:
print(df["control_zone_name"].unique())


<StringArray>
['CSWS',  'EDE', 'GRDA', 'INDN', 'KACY', 'KCPL',  'LES',  'MPS', 'NPPD',
 'OKGE', 'OPPD', 'SECI', 'SPRM',  'SPS', 'WAUE', 'WFEC',   'WR', 'PRPA',
 'WACM', 'WAUW']
Length: 20, dtype: str


In [68]:
print(df["forecast_area_type"].unique())

<StringArray>
['CF', 'NC']
Length: 2, dtype: str
